## Emlearn model on TinyML

##### Step 1:

- Create a classical machine learning model using decision tree
- Convert the model into a header (.h) file
- 

In [27]:
# 1. Install emlearn
!pip install -q emlearn scikit-learn numpy

import emlearn
import numpy as np
from sklearn.tree import DecisionTreeClassifier

# 2. Create and train a simple model (AND Gate example)
X = np.array([[0, 0], [255, 0], [0, 255], [255, 255]], dtype=np.int16)
y = np.array([0, 0, 0, 1])

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X, y)

# 3. Convert to C header
# The 'inline' method is fastest and requires the least RAM
cmodel = emlearn.convert(model, method='inline')
cmodel.save(file='my_model.h', name='my_model')

# 4. Find where emlearn headers are located so you can download them
print(f"Download the core headers from: {emlearn.includedir}")


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


Download the core headers from: /home/user/.local/lib/python3.12/site-packages/emlearn


##### Step 2: 

- Create a C based inference code to use the model stored in the header file
- Prepare  input_feature to be be used by the model 
- Call **model_name_predict** function to execute the model and store the value in variable result
- Print the inference output from the model

In [37]:
%%writefile my_emlearn_model.c

#include <stdio.h>
#include <stdint.h>

// 1. Include the model generated in Colab
#include "my_model.h"

int main() {
    printf("--- emlearn Pure C Inference ---\n");

    // 2. Prepare input features (matching the training data range)
    int16_t input_features[2] = {255, 255};

    // 3. Call the generated prediction function
    // Format: [name]_predict(features_array, num_features)
    int32_t result = my_model_predict(input_features, 2);

    printf("Input: [255, 255] -> AI Prediction: %d\n", result);

    return 0;
}

Overwriting my_emlearn_model.c


In [38]:
!gcc -o ml_model my_emlearn_model.c -I $(python3 -c "import emlearn; print(emlearn.includedir)")

In [39]:
!./ml_model

--- emlearn Pure C Inference ---
Input: [255, 255] -> AI Prediction: 1


In [0]:
import os

# Path to the generated C header file
model_file_path = 'my_model.h'

# Check if the file exists and get its size
if os.path.exists(model_file_path):
    size_bytes = os.path.getsize(model_file_path)
    print(f"Estimated model size (my_model.h): {size_bytes} bytes")
else:
    print(f"Error: Model file '{model_file_path}' not found.")


NXP eIQ ONNX2TFLite: Reference https://github.com/NXP/eiq-onnx2tflite

In [0]:
%%writefile requirements.txt
flatbuffers==24.3.25 # TensorFlow dependency
numpy
protobuf
onnx~=1.17.0
onnxruntime~=1.21.1
sympy
deprecated

In [0]:
%%bash

pip install -r requirements.txt
pip install --index-url https://eiq.nxp.com/repository/ eiq-onnx2tflite

Repository of ONNX Models : https://github.com/onnx/models


In [0]:
!wget https://github.com/onnx/models/raw/main/Computer_Vision/mobilenetv2_140_Opset18_timm/mobilenetv2_140_Opset18.onnx

In [0]:
import onnx2tflite.src.converter.convert as convert

binary_tflite_model = convert.convert_model("mobilenetv2_140_Opset18.onnx")

with open("model_mobilev2.tflite", "wb") as f:
    f.write(binary_tflite_model)

In [0]:
%%bash

# Install the Moonshine Voice library and its dependencies
pip install moonshine-voice transformers torchaudio ffmpeg-python


In [0]:
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode
import io
import ffmpeg

# JavaScript to handle the browser microphone
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})

var record = time => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  const chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async () => {
    const blob = new Blob(chunks)
    const text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record_audio(filename='recording.wav', seconds=3):
    print(f"Recording for {seconds} seconds...")
    display(Javascript(RECORD_JS))

    # Execute JS and get base64 string
    s = output.eval_js(f'record({seconds * 1000})')
    b = b64decode(s.split(',')[1])

    # Use ffmpeg to convert to 16kHz Mono (Moonshine's requirement)
    process = (
        ffmpeg
        .input('pipe:0')
        .output(filename, ar='16000', ac='1')
        .run_async(pipe_stdin=True, quiet=True, overwrite_output=True)
    )
    process.communicate(input=b)

    print(f"Saved to {filename}")
    return filename

In [0]:
# 2. Record 5 seconds of audio
audio_file = record_audio(seconds=5)

# 3. Transcribe with Moonshine
from transformers import pipeline
transcriber = pipeline("automatic-speech-recognition", model="UsefulSensors/moonshine-tiny")

print("Processing transcript...")
result = transcriber(audio_file)
print(f"\nResult: {result['text']}")

In [0]:
!pip install -q transformers torch pillow

In [0]:
from transformers import pipeline
from PIL import Image
import requests

# Load the ViT pipeline
classifier = pipeline("image-classification", model="google/vit-base-patch16-224")

# Load an image (e.g., a cat)
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# Predict
results = classifier(image)
for result in results:
    print(f"{result['label']}: {round(result['score'], 4)}")

In [0]:
import torch
import matplotlib.pyplot as plt
from transformers import ViTImageProcessor, ViTForImageClassification
from PIL import Image
import requests

# 1. Load model and processor
model_name = 'google/vit-base-patch16-224'
processor = ViTImageProcessor.from_pretrained(model_name)
model = ViTForImageClassification.from_pretrained(model_name, output_attentions=True)

# 2. Get an image
url = 'http://images.cocodataset.org/val2017/000000039769.jpg' # Cat image
image = Image.open(requests.get(url, stream=True).raw)
inputs = processor(images=image, return_tensors="pt")

# 3. Forward pass with attention output
outputs = model(**inputs, output_attentions=True)

# 4. Extract attention from the last layer
# Shape: [batch, num_heads, sequence_length, sequence_length]
attentions = outputs.attentions[-1]

# Average across all attention heads
att_mat = torch.mean(attentions, dim=1).squeeze()

# The first row represents the CLS token's attention to all other 196 patches
# We ignore the first element (CLS to itself) and reshape the rest to 14x14
cls_attention = att_mat[0, 1:].reshape(14, 14).detach().numpy()

# 5. Visualize
plt.imshow(image)
plt.imshow(cls_attention, cmap='jet', alpha=0.5, extent=(0, 224, 224, 0))
plt.title("ViT Attention Map (Last Layer)")
plt.axis('off')
plt.show()

In [0]:
# Quick Colab demo for DINO (Self-Supervised ViT)
import torch
from PIL import Image
import requests

# Load DINO ViT-Small (pretrained without labels)
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits16')
model.eval()

# Use this to visualize how the model automatically segments objects
# from the background without ever being told what a "cat" or "car" is.

In [0]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import requests
import matplotlib.pyplot as plt
import numpy as np

# Load DINO ViT-Small (8x8 patches for higher resolution detail)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = torch.hub.load('facebookresearch/dino:main', 'dino_vits8').to(device)
model.eval()

# Preprocessing: DINO expects 224x224 or multiples of the patch size (8)
transform = T.Compose([
    T.Resize(480), # High res for better visualization
    T.CenterCrop(480),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# Load an image (Change the URL to your own or use a local path)
url = "https://dl.fbaipublicfiles.com/dino/img.png"
img_raw = Image.open(requests.get(url, stream=True).raw).convert('RGB')
img_tensor = transform(img_raw).unsqueeze(0).to(device)

In [0]:
# 1. Forward pass to get attention
# We use 'get_last_selfattention' which is a built-in method for the DINO hub model
with torch.no_grad():
    attentions = model.get_last_selfattention(img_tensor)

# 2. Process the attention map
# Structure: [Batch, Heads, Num_Patches+1, Num_Patches+1]
nh = attentions.shape[1] # Number of attention heads (usually 6 for ViT-S)
w_featmap = img_tensor.shape[-2] // 8
h_featmap = img_tensor.shape[-1] // 8

# We keep only the attention of the [CLS] token to the other patches
# Index 0 is the CLS token; we take 0, :, 0, 1:
attentions = attentions[0, :, 0, 1:].reshape(nh, w_featmap, h_featmap)

# 3. Visualize the Different "Heads"
fig, axs = plt.subplots(1, nh + 1, figsize=(20, 5))
axs[0].imshow(img_raw.resize((480, 480)))
axs[0].set_title("Original")
axs[0].axis('off')

for i in range(nh):
    axs[i+1].imshow(attentions[i].cpu().numpy(), cmap='magma')
    axs[i+1].set_title(f"Head {i}")
    axs[i+1].axis('off')

plt.show()

In [0]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from PIL import Image
import requests
import torchvision.transforms as T

# 1. Load Model (DINOv2 is highly recommended for this)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14').to(device)
model.eval()

# 2. Load and Preprocess Image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = Image.open(requests.get(url, stream=True).raw).convert('RGB')
w, h = img.size
# Resize to a multiple of patch size (14)
img_t = T.Compose([
    T.Resize((448, 448)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])(img).unsqueeze(0).to(device)

# 3. Extract Patch Features
with torch.no_grad():
    features_dict = model.forward_features(img_t)
    features = features_dict['x_norm_patchtokens'] # Shape: [1, 1024, 384]

# 4. Perform PCA (Reduce 384-D -> 3-D)
# Flatten to [Num_Patches, Embedding_Dim]
features = features.squeeze(0).cpu().numpy()
pca = PCA(n_components=3)
pca_features = pca.fit_transform(features)

# 5. Normalize and Reshape to RGB Image
# Scale features to 0-1 range for RGB visualization
pca_features = (pca_features - pca_features.min()) / (pca_features.max() - pca_features.min())

# Reshape back to grid (448/14 = 32 patches)
grid_size = 448 // 14
pca_img = pca_features.reshape(grid_size, grid_size, 3)

# 6. Plot Results
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(img.resize((448, 448)))
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(pca_img)
plt.title("DINO PCA Segmentation")
plt.axis('off')
plt.show()

In [0]:
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import umap

X, y = make_classification(n_samples=100000, n_features=20, n_classes=5, n_informative=5, random_state=0)
X_train, _, y_train, _ = train_test_split(X, y, test_size=0.2)

umap_model = umap.UMAP(n_neighbors=15, n_components=2, random_state=42, min_dist=0.0, n_jobs=1)
X_train_umap = umap_model.fit_transform(X_train)
y_train
# Plot the UMAP result
plt.figure(figsize=(10, 8))
plt.scatter(X_train_umap[:, 0], X_train_umap[:, 1], c=y_train, cmap='Spectral', s=10)
plt.colorbar(label="Activity")
plt.title("UMAP projection")
plt.xlabel("UMAP Component 1")
plt.ylabel("UMAP Component 2")
plt.show()

In [0]:
!pip install -q umap-learn seaborn palmerpenguins

In [0]:
import umap
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from palmerpenguins import load_penguins

# 1. Load the "Relatable" Dataset
penguins = load_penguins()

# 2. Preprocessing
# Drop rows with missing values (real-world data has NaNs!)
penguins = penguins.dropna()

# Select numeric features: Bill length, Bill depth, Flipper length, Body mass
data = penguins[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]

# 3. Scaling is MANDATORY for UMAP
# Because 'body_mass_g' is in the thousands while 'bill_depth' is < 25.
scaled_data = StandardScaler().fit_transform(data)

# 4. Fit UMAP
# n_neighbors: balances local vs global structure (15 is a good default)
# min_dist: how tightly points are packed (0.1 is standard)
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(scaled_data)

# 5. Visualization
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x=embedding[:, 0],
    y=embedding[:, 1],
    hue=penguins.species,
    palette='viridis',
    s=60,
    alpha=0.8
)

plt.title('UMAP Projection of Palmer Penguins', fontsize=16)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')
plt.legend(title='Species', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [0]:
!pip install -q https://github.com/roboflow/rf-detr/archive/refs/tags/1.5.0.rc1.zip

In [0]:
import os

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Scale data-loader workers with available CPUs so the GPU is kept fed.
num_workers = max(os.cpu_count(), 2)
print(f"Data loader workers: {num_workers}")

In [0]:
!pip uninstall transformers
!pip install transformers==4.50.0

In [0]:
import requests
import supervision as sv
from PIL import Image
from rfdetr import RFDETRNano
from rfdetr.util.coco_classes import COCO_CLASSES

model = RFDETRNano()

image = "https://media.roboflow.com/notebooks/examples/dog-2.jpeg"
detections = model.predict(image, threshold=0.5)

labels = [f"{COCO_CLASSES[class_id]}" for class_id in detections.class_id]

annotated_image = sv.BoxAnnotator().annotate(image, detections)
annotated_image = sv.LabelAnnotator().annotate(annotated_image, detections, labels)

In [0]:
import cv2
import requests
import supervision as sv
from PIL import Image
from io import BytesIO
from rfdetr import RFDETRBase

# 1. Load the pre-trained Base model (trained on COCO)
model = RFDETRBase()

# 2. Load an image from a URL
url = "https://media.roboflow.com/notebooks/examples/dog-2.jpeg"
response = requests.get(url)
image = Image.open(BytesIO(response.content)).convert("RGB")

# 3. Predict
# Returns a supervision Detections object
detections = model.predict(image, threshold=0.5)

# 4. Annotate and Plot
box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

annotated_image = box_annotator.annotate(scene=image.copy(), detections=detections)
annotated_image = label_annotator.annotate(scene=annotated_image, detections=detections)

sv.plot_image(annotated_image)

In [0]:
import torch
from PIL import Image
import requests
from transformers import AutoImageProcessor, ViTModel

# 1. Load the model and processor
model_name = "google/vit-base-patch16-224-in21k"
processor = AutoImageProcessor.from_pretrained(model_name)
model = ViTModel.from_pretrained(model_name)

# 2. Load and preprocess an image
url = 'http://images.cocodataset.org/val2017/000000039769.jpg'
image = Image.open(requests.get(url, stream=True).raw)
inputs = processor(images=image, return_tensors="pt")

# 3. Forward pass
with torch.no_grad():
    outputs = model(**inputs)

# 4. Extract the [CLS] token
# last_hidden_state shape: [batch_size, sequence_length, hidden_size]
# sequence_length = (number_of_patches + 1) -> 196 + 1 = 197
last_hidden_state = outputs.last_hidden_state
cls_token = last_hidden_state[:, 0, :]

print(f"Full Hidden State Shape: {last_hidden_state.shape}")
print(f"Extracted [CLS] Token Shape: {cls_token.shape}")

In [0]:
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModel
import numpy as np

# Load DINOv2 (Small version is fastest for Colab)
device = "cuda" if torch.cuda.is_available() else "cpu"
processor = AutoImageProcessor.from_pretrained("facebook/dinov2-small")
model = AutoModel.from_pretrained("facebook/dinov2-small").to(device)

@torch.no_grad()
def get_embedding(image_path):
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)
    outputs = model(**inputs)
    # The [CLS] token is the first token in the last hidden state
    embedding = outputs.last_hidden_state[:, 0, :]
    return embedding.cpu().numpy().flatten()

In [0]:
import os
os.environ["KERAS_BACKEND"] = "jax" # JAX is recommended for Transformers in 2026

import keras_hub
import keras
import numpy as np
from PIL import Image

# Print available presets to debug the error
print("Available ObjectDetector presets:", keras_hub.models.ObjectDetector.presets.keys())

# 1. Load a SOTA Transformer-based Object Detector
# Preset "dfine_s_obj2coco" is a Small D-FINE model pretrained on Objects365 & COCO
detector = keras_hub.models.ObjectDetector.from_preset(
    "retinanet_resnet50_fpn_coco", # Changed to a valid preset
    bounding_box_format="xywh"
)

# 2. Load and Preprocess an Image
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/c/c4/Savannah_Cat_portrait.jpg/640px-Savannah_Cat_portrait.jpg"
image_path = keras.utils.get_file(origin=url)
image = keras.utils.load_img(image_path, target_size=(640, 640))
image_array = keras.utils.img_to_array(image)

# 3. Run Inference
# The output contains 'boxes', 'confidence', and 'classes'
outputs = detector.predict(np.expand_dims(image_array, axis=0))

# 4. Access the Detections
boxes = outputs["boxes"][0]
classes = outputs["classes"][0]
confidences = outputs["confidence"][0]

print(f"Detected {len(boxes[confidences > 0.5])} objects with >50% confidence.")

In [0]:
!pip install -U keras-hub
!pip install -U keras

In [0]:
!pip install keras-cv
import torch
import torch.nn as nn
import torchvision.transforms as T
from PIL import Image
import requests
import matplotlib.pyplot as plt
import numpy as np

import os
os.environ["KERAS_BACKEND"] = "jax" # JAX is recommended for Transformers in 2026

import keras_hub
import keras
import keras_cv # Added import for keras_cv
import numpy as np
from PIL import Image

# Pretrained DINOV3 model.
input_data = {
    "pixel_values": np.ones(shape=(1, 224, 224, 3), dtype="float32"), # Changed to 224x224
}
model = keras_hub.models.DINOV3Backbone.from_preset(
    "dinov3_vit_small_lvd1689m"
)
model(input_data)

# Pretrained DINOV3 model with custom image shape.
input_data = {
    "pixel_values": np.ones(shape=(1, 224, 224, 3), dtype="float32"), # Changed key from 'images' to 'pixel_values'
}
model = keras_hub.models.DINOV3Backbone.from_preset(
    "dinov3_vit_small_lvd1689m", image_shape=(224, 224, 3)
)
model(input_data)

# Randomly initialized DINOV3 model with custom config.
model = keras_hub.models.DINOV3Backbone(
    patch_size=14,
    num_layers=2,
    hidden_dim=32,
    num_heads=2,
    intermediate_dim=128,
    image_shape=(224, 224, 3),
)
model(input_data)

# Accessing feature pyramid outputs.
backbone = keras_hub.models.DINOV3Backbone.from_preset(
    "dinov3_vit_small_lvd1689m", image_shape=(224, 224, 3)
)
model = keras.Model(
    inputs=backbone.inputs,
    outputs=backbone.pyramid_outputs,
)
features = model(input_data)

print (features)


In [0]:
!pip install ai-edge-litert

In [0]:
!pip install -U -q keras-hub

In [0]:
import numpy as np
import cv2
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter
from tensorflow.keras.applications.resnet50 import ResNet50
import keras_hub

# 1. Load the model from Keras Hub
model = keras_hub.models.ImageClassifier.from_preset(
    "resnet_50_imagenet",
    activation="softmax"
)

# 2. Initialize the converter
# from_keras_model is the recommended entry point for TF2/TF3 backends
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 3. Optional: Post-Training Quantization (reduces size by ~4x)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 4. Convert and Save
tflite_model = converter.convert()
with open("resnet50.tflite", "wb") as f:
    f.write(tflite_model)

# 1. Load ViT model
# Note: ViT usually requires a specific 224x224 input
vit_model = keras_hub.models.ImageClassifier.from_preset(
    "vit_base_patch16_224_imagenet",
    activation="softmax"
)

converter = tf.lite.TFLiteConverter.from_keras_model(vit_model)

# 2. Enable 'Select TF Ops' for Transformer compatibility
# This allows TFLite to 'fall back' to regular TensorFlow for complex attention ops
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS, # Enable standard TFLite ops
    tf.lite.OpsSet.SELECT_TF_OPS    # Enable TensorFlow ops fallback
]

# 3. Convert and Save
tflite_vit_model = converter.convert()
with open("vit_base.tflite", "wb") as f:
    f.write(tflite_vit_model)


In [0]:
import numpy as np
import cv2
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter
import keras_hub
import matplotlib.pyplot as plt

def create_and_show_obfuscated(image_path, patch_size=32):
    # 1. Load and prepare image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224)) # Standard ViT size

    h, w, c = img.shape
    patches = []

    # 2. Break into patches
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            patches.append(img[i:i+patch_size, j:j+patch_size])

    # 3. Shuffle patches
    # This is what "confuses" the local filters of a CNN (ResNet)
    np.random.seed(42) # For reproducible results
    np.random.shuffle(patches)

    # 4. Reconstruct the obfuscated image
    obfuscated = np.zeros_like(img)
    idx = 0
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            obfuscated[i:i+patch_size, j:j+patch_size] = patches[idx]
            idx += 1

    # 5. Display side-by-side
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Original Image (ResNet Input)")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(obfuscated)
    plt.title(f"Obfuscated - {patch_size}px Patches (ViT Input)")
    plt.axis("off")

    plt.show()



def obfuscate_image(image, patch_size=32):
    """Shuffles patches of the image to obfuscate it."""
    h, w, c = image.shape
    patches = []
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            patches.append(image[i:i+patch_size, j:j+patch_size])

    np.random.shuffle(patches)

    # Reconstruct the image
    obfuscated = np.zeros_like(image)
    idx = 0
    for i in range(0, h, patch_size):
        for j in range(0, w, patch_size):
            obfuscated[i:i+patch_size, j:j+patch_size] = patches[idx]
            idx += 1
    return obfuscated



def run_tflite_inference(model_path, image):
    """Standard TFLite inference loop."""
    interpreter = Interpreter(model_path=model_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Resize and normalize
    target_size = input_details[0]['shape'][1:3]
    resized = cv2.resize(image, (target_size[1], target_size[0]))
    input_data = np.expand_dims(resized, axis=0).astype(np.float32) / 255.0

    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    return interpreter.get_tensor(output_details[0]['index'])[0]

# --- Main Logic ---
# Using an available image from the 'giant_panda' folder
image_path = "pizza.png"
image = cv2.imread(image_path)

# Check if image was loaded correctly
if image is None:
    print(f"Error: Could not load image from {image_path}")
else:
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    # Resize image_rgb to be a multiple of patch_size (56) for obfuscation
    target_dim = 224 # 224 is 4 * 56
    image_rgb_resized = cv2.resize(image_rgb, (target_dim, target_dim))

    # 1. Classify with ResNet (CNN)
    # Pass the resized image for consistent input dimensions
    resnet_out = run_tflite_inference("resnet50.tflite", image_rgb_resized)
    print(f"ResNet Original Top Class: {np.argmax(resnet_out)}")

    # 2. Obfuscate
    obfuscated_rgb = obfuscate_image(image_rgb_resized, patch_size=56)
    create_and_show_obfuscated(image_path, patch_size=56)

    # 3. Classify with ViT (Transformer)
    vit_out = run_tflite_inference("vit_base.tflite", obfuscated_rgb)
    print(f"ViT Obfuscated Top Class: {np.argmax(vit_out)}")

https://www.kaggle.com/models/keras/d-fine/keras/dfine_nano_coco/1

In [0]:
import keras
import keras_hub
import numpy as np
from keras_hub.models import DFineBackbone
from keras_hub.models import DFineObjectDetector
from keras_hub.models import HGNetV2Backbone
import cv2 # Added for cv2.imread, cvtColor, resize

object_detector = DFineObjectDetector.from_preset(
    "dfine_nano_coco"
)

# Create a random image.
#image = np.random.uniform(size=(1, 256, 256, 3)).astype("float32")

image_path = "pizza.png"
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img = cv2.resize(img, (256, 256)) # Standard ViT size

# Add a batch dimension to the image
img_batch = np.expand_dims(img, axis=0)

# Make predictions.
predictions = object_detector.predict(img_batch)

# The output is a dictionary containing boxes, labels, confidence scores,
# and the number of detections.
print(predictions["boxes"].shape)
print(predictions["labels"].shape)
print(predictions["confidence"].shape)
print(predictions["num_detections"])


In [0]:
num_detections = predictions["num_detections"][0]
detected_labels = predictions["labels"][0][:num_detections]
detected_confidences = predictions["confidence"][0][:num_detections]

print("Detected Objects and Confidences:")
for i in range(num_detections):
    # Assuming COCO_CLASSES is available from a previous cell, as in other examples in the notebook
    # If COCO_CLASSES is not defined, this will raise an error. For now, we assume it exists.
    label_name = f"Class {detected_labels[i]}" # Fallback if COCO_CLASSES not available
    try:
        from rfdetr.util.coco_classes import COCO_CLASSES
        label_name = COCO_CLASSES[int(detected_labels[i])]
    except (ImportError, KeyError):
        pass # Keep fallback if import fails or class ID is not in COCO_CLASSES

    print(f"- {label_name}: {detected_confidences[i]:.4f}")

In [0]:
!pip install --upgrade mqtt-spb-wrapper paho-mqtt

In [0]:
%%writefile sparkplug_spb_node.py

import time
import paho.mqtt.client as mqtt
from mqtt_spb_wrapper import MqttSpbEntityEdgeNode

# --- Configuration ---
# Use 'broker.hivemq.com' for public testing or your HiveMQ Cloud host
_BROKER_HOST = "broker.hivemq.com"
_GROUP_ID = "Sparkplug_Experiment"
_NODE_ID = "Python_Node_01"
_MQTT_PORT = 1883 # Default MQTT port

def experiment_spb():
    # 1. Create the Edge Node
    # The wrapper handles the Sparkplug B topic structure automatically
    node = MqttSpbEntityEdgeNode(_GROUP_ID, _NODE_ID, _BROKER_HOST)

    # Explicitly set host and port to override any internal defaults
    node.host = _BROKER_HOST
    node.port = _MQTT_PORT

    # 2. Define Metrics (The "Birth Certificate")
    # These declare what data this node will provide to the SCADA system
    node.attributes.set_value("Description", "Python Simulation Node")
    node.data.set_value("Temperature", 25.5)
    node.data.set_value("Status_LED", False)

    # 3. Connect to HiveMQ
    # This automatically sends the NBIRTH (Node Birth) message
    print(f"Connecting to {node.host}:{node.port}...") # Updated print statement
    if node.connect() != 0:
        print("Connection failed!")
        return

    print("NBIRTH published. Node is now online.")

    # 4. Data Update Loop (NDATA)
    try:
        temp = 25.5
        while True:
            temp += 0.5  # Simulate rising temperature

            # Update the metric value locally
            node.data.set_value("Temperature", temp)

            # Publish NDATA (only changed values are sent - "Report by Exception")
            node.publish_data()
            print(f"NDATA Sent: Temperature = {temp}")

            time.sleep(5)
    except KeyboardInterrupt:
        # 5. Disconnect
        # This allows the node to go offline gracefully
        node.disconnect()
        print("Disconnected.")

if __name__ == "__main__":
    experiment_spb()

In [0]:
import socket

def check_mqtt_broker_connectivity(host, port):
    try:
        # Create a socket object
        s = socket.create_connection((host, port), timeout=5)
        s.close()
        print(f"Successfully connected to {host}:{port}")
        return True
    except socket.error as err:
        print(f"Could not connect to {host}:{port}. Error: {err}")
        return False

# --- Configuration from sparkplug_spb_node.py ---
_BROKER_HOST = "broker.hivemq.com"
_MQTT_PORT = 1883 # Default MQTT port

print(f"Attempting to connect to MQTT broker {_BROKER_HOST}:{_MQTT_PORT}...")
check_mqtt_broker_connectivity(_BROKER_HOST, _MQTT_PORT)

In [0]:
!python sparkplug_spb_node.py &

In [0]:
%%writefile sparkplug_spb_scada.py

import paho.mqtt.client as mqtt
from mqtt_spb_wrapper import MqttSpbEntityScada

def on_message(client, userdata, msg):
    # This function decodes the binary 'msg.payload' into a Python dictionary
    decoded = MqttSpbEntityScada.decode_payload(msg.payload)
    print(f"Topic: {msg.topic}\nData: {decoded}\n")

client = mqtt.Client()
client.on_message = on_message
client.connect("broker.hivemq.com", 1883)
client.subscribe("spBv1.0/Sparkplug_Experiment/#") # Subscribe to all Sparkplug B messages
client.loop_forever()